# Endpoint de inferencia del peaje M1 + M2 + M3 - Severich


## 1. Código y pesos

In [1]:
from google.colab import files, drive
import os, zipfile, shutil
up = files.upload()
with zipfile.ZipFile(next(iter(up))) as z: z.extractall('/content')
CARPETA = '/content/inferencia' if os.path.isdir('/content/inferencia') else '/content/inferencia/inferencia'
os.chdir(CARPETA)
try:
    drive.mount('/content/drive')
    for nombre in ('m2_vehiculos.pt', 'm3_llantas.pt'):
        origen = '/content/drive/MyDrive/peaje/' + nombre
        if os.path.exists(origen):
            shutil.copy(origen, CARPETA + '/modelos/' + nombre); print('Pesos propios:', nombre)
        else:
            print('Sin', nombre, '-> se usa el método sin entrenar (YOLOE)')
except Exception as e:
    print('Drive no montado, se usan métodos sin entrenar:', e)
print('Carpeta:', os.getcwd(), os.listdir('.'))

Saving inferencia (3).zip to inferencia (3).zip
Mounted at /content/drive
Pesos propios: m2_vehiculos.pt
Pesos propios: m3_llantas.pt
Carpeta: /content/inferencia ['modelos', 'diagnostico.py', 'app.py', 'evaluar_pipeline.py', 'modelo', 'm1_placas.py', 'evaluar_ocr.py', 'config.py', 'README.md', 'm3_ejes.py', 'tests', 'colab_servidor.ipynb', 'requirements.txt', 'm2_vehiculos.py', 'ejes_logic.py']


## 2. Dependencias

In [2]:
!pip -q install -r requirements.txt
!pip -q install 'git+https://github.com/ultralytics/CLIP.git'
# !pip -q install easyocr

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-semantic-conventions 0.63b1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
opentelemetry-sdk 1.42.1 requires opentelemetry-api==1.42.1, but you have opentelemetry-api 1.45.2 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 3. Diagnóstico del entorno

In [3]:
!python diagnostico.py | tee diagnostico_colab.txt

Python: 3.13.16 | Linux-6.6.159+-x86_64-with-glibc2.39
torch: 2.14.1+cu130
torchvision: 0.29.1+cu130
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
ultralytics: 8.4.218
onnxruntime: 1.29.0
fast-plate-ocr: instalado
easyocr: NO DISPONIBLE (ModuleNotFoundError)
opencv: 4.14.0
numpy: 2.4.6
fastapi: 0.165.0
uvicorn: 0.54.0
CUDA disponible: True | CUDA build: 13.0
GPU: Tesla T4


## 4. Configuración y carga de modelos

In [4]:
import os, secrets, sys
from google.colab import userdata
try:
    os.environ['INFERENCE_API_KEY'] = userdata.get('INFERENCE_API_KEY')
except Exception:
    os.environ['INFERENCE_API_KEY'] = secrets.token_urlsafe(24)
    print('⚠ No hay secreto INFERENCE_API_KEY: se generó una clave temporal (cambia en cada ejecución).')
os.environ.setdefault('OCR_ENGINE', 'fastplate')
sys.path.insert(0, os.getcwd())
import app as servidor
m = servidor.cargar_modelos()
print('M2:', m['m2'].nombre); print('M3:', m['m3'].nombre)

M2: propio (YOLO-seg entrenado)
M3: propio (YOLO-pose entrenado)


## 5. Servidor + túnel

In [5]:
import uvicorn, threading, time, subprocess, re, urllib.request
if 'servidor_http' in globals() and servidor_http.started:
    print('El servidor ya está corriendo.')
else:
    servidor_http = uvicorn.Server(uvicorn.Config(servidor.app, host='0.0.0.0', port=8000, log_level='warning'))
    threading.Thread(target=servidor_http.run, daemon=True).start()
    while not servidor_http.started: time.sleep(0.2)
if 'URL_PUBLICA' not in globals():
    !wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /usr/local/bin/cloudflared
    tunel = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--url','http://localhost:8000'],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    log = []
    threading.Thread(target=lambda: [log.append(l) for l in tunel.stdout], daemon=True).start()
    URL_PUBLICA = None
    for _ in range(80):
        mm = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', ''.join(log))
        if mm: URL_PUBLICA = mm.group(0); break
        time.sleep(0.5)
    time.sleep(4)
print('\nPega esto en el .env del servidor web y reinícialo (sudo systemctl restart peaje):\n')
print('INFERENCE_URL=' + str(URL_PUBLICA)); print('INFERENCE_API_KEY=<la misma clave del secreto>')


Pega esto en el .env del servidor web y reinícialo (sudo systemctl restart peaje):

INFERENCE_URL=https://jennifer-command-comparisons-tank.trycloudflare.com
INFERENCE_API_KEY=<la misma clave del secreto>


## 6. Prueba a través del túnel

In [6]:
import requests, json
IMG = 'ejemplo.jpg'
if not os.path.exists(IMG): upl = files.upload(); IMG = next(iter(upl))     # sube una foto de un vehículo con placa
r = requests.post(URL_PUBLICA + '/predict', files={'file': open(IMG,'rb')}, headers={'X-API-Key': os.environ['INFERENCE_API_KEY']}, timeout=120)
d = r.json(); d.pop('imagen_anotada', None); print(r.status_code); print(json.dumps(d, indent=2, ensure_ascii=False))

Saving photo_2_2026-11-22_22-38-33.jpg to photo_2_2026-11-22_22-38-33.jpg
200
{
  "placa": {
    "texto": "4257YTT",
    "confianza": 0.958,
    "bbox": [
      1923,
      1124,
      2264,
      1375
    ],
    "valida": true,
    "crudo": "4257YTT"
  },
  "vehiculo": {
    "categoria": "minibus",
    "confianza": 0.912,
    "area_mascara": 0.382
  },
  "ejes": null,
  "vista": "-",
  "nota_ejes": "Sin llantas detectadas",
  "keypoints": [],
  "latencia_ms": 3903.3,
  "tiempos_ms": {
    "m1": 64.1,
    "m2": 3678.5,
    "m3": 160.7
  }
}


## LLM del agente conversacional: Qwen con Ollama


In [7]:
MODELO_LLM = 'qwen2.5:7b'
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess, time, re, threading, requests
env_ollama = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_KEEP_ALIVE='-1')
ollama = subprocess.Popen(['ollama', 'serve'], env=env_ollama, stdout=open('/content/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(40):
    try: requests.get('http://127.0.0.1:11434/api/tags', timeout=2); break
    except Exception: time.sleep(1)
!ollama pull $MODELO_LLM

t0 = time.time()
r = requests.post('http://127.0.0.1:11434/api/generate', json={'model': MODELO_LLM, 'prompt': 'ok', 'stream': False, 'keep_alive': -1}, timeout=600)
print('Modelo cargado en la GPU en', round(time.time() - t0), 's ->', r.status_code)
tunel_llm = subprocess.Popen(['cloudflared','tunnel','--no-autoupdate','--http-host-header','localhost:11434','--url','http://localhost:11434'],
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
log_llm = []
threading.Thread(target=lambda: [log_llm.append(l) for l in tunel_llm.stdout], daemon=True).start()
URL_LLM = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', ''.join(log_llm))
    if m: URL_LLM = m.group(0); break
print('Pega en el panel -> Modelo de lenguaje del agente:')
print('  URL base :', URL_LLM + '/v1')
print('  Modelo   :', MODELO_LLM)
print('  API key  :')

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.

Modelo cargado en la GPU en 171 s -> 200
Pega en el panel -> Modelo de lenguaje del agente:
  URL base : https://massive-firmware-maiden-reggae.trycloudflare.com/v1
  Modelo   : qwen2.5:7b
  API key  : (déjala vacía)


In [8]:
time.sleep(5)
r = requests.post(URL_LLM + '/v1/chat/completions', json={'model': MODELO_LLM, 'messages': [{'role': 'user', 'content': 'Responde solo: ok'}]}, timeout=90)
print(r.status_code, r.json()['choices'][0]['message']['content'])

200 ok


## 7. Detener


In [9]:
# DETENER = False
# if DETENER:
#     servidor_http.should_exit = True; tunel.terminate(); print('Detenido')
# else:
#     print('Sigue activo. Pon DETENER = True para apagarlo.')

In [10]:
import os, shutil
mod = CARPETA + '/modelos/m2_vehiculos.pt'
if os.path.exists(mod):
    shutil.move(mod, CARPETA + '/modelos/m2_vehiculos_entrenado.pt')
servidor._modelos.clear()
modelos = servidor.cargar_modelos()
print(modelos['m2'].nombre)
print(modelos['m3'].nombre)

/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


YOLOE sin entrenar (reemplazar por M2 propio)
propio (YOLO-pose entrenado)


In [11]:
shutil.move(CARPETA + '/modelos/m2_vehiculos_entrenado.pt', mod)
servidor._modelos.clear(); servidor.cargar_modelos()

{'m1': <m1_placas.M1 at 0x7b99d4303250>,
 'm2': <m2_vehiculos.M2 at 0x7b99d43039d0>,
 'm3': <m3_ejes.M3 at 0x7b9a04957250>}

In [16]:
import csv, cv2, sys
sys.path.insert(0, CARPETA)
import evaluar_pipeline as ev
filas = list(csv.DictReader(open('/content/drive/MyDrive/peaje/etiquetas.csv', encoding='utf-8-sig')))
print(ev.informe(ev.evaluar(filas, '/content/drive/MyDrive/peaje/fotos',
                            servidor.analizar, lambda p: cv2.imread(str(p)))))

Imágenes evaluadas: 63

| Métrica | Resultado |
|---|---|
| Placa exacta (M1) | 49/63 (77.8 %), sin lectura: 3 |
| Categoría correcta (M2) | 40/63 (63.5 %) |
| Ejes exactos (M3) | 39/63 (61.9 %), no determinables: 24 |
| Error medio de ejes (MAE) | 0.76 ejes |
| Latencia de inferencia | media 130 ms · p95 158 ms · máx 653 ms |

Matriz de confusión de categoría (real → predicha):
  auto → auto: 26
  auto → bus: 2
  auto → minibus: 1
  microbus → bus: 7
  minibus → auto: 10
  minibus → bus: 2
  minibus → camion: 1
  minibus → minibus: 14

Placas falladas (archivo, real, obtenido):
  - 1.jpg: 5003BX → 5003BXC
  - 16.jpg: 124ZHP → (nada)
  - 17.jpg: 4457NIP → 1085FBC
  - 17.png: 195SHO → 195SHC
  - 18.jfif: 2906DRF → 2906DFR
  - 19.jpg: 526PLO → 526PLD
  - 21.jpg: 24830OLH → 2483DLH
  - 24.jpg: 3100IGO → 3101AFU
  - 27.jpg: 3624PHG → 3624RHG
  - 3.png: 4413HLH → 4413HHH
  - 37.jfif: 6672FAN → 6672FNN
  - 4.jpg: 3001CTA → 300ICT
  - 6.jfif: 4367GGX → (nada)
  - photo_3_2026-11-22_23-37-42.j

Existe el modelo entrenado antes: True
M2 ahora: YOLOE sin entrenar (reemplazar por M2 propio)
